In [1]:
import pandas as pd
import re
import kagglehub

/Users/ashleshkanchan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ashleshkanchan/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")

print("Path to dataset files:", path)

100%|██████████| 41.0M/41.0M [00:05<00:00, 8.59MB/s]

Extracting files...


Path to dataset files: /Users/ashleshkanchan/.cache/kagglehub/datasets/emineyetm/fake-news-detection-datasets/versions/1


In [10]:
fake_df = pd.read_csv(path + '/News _dataset/Fake.csv')
true_df = pd.read_csv(path + '/News _dataset/True.csv')

In [11]:
print(fake_df.head())
print(true_df.head())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept t

In [12]:
fake_df["label"] = 0
true_df["label"] = 1

In [13]:
df = pd.concat([fake_df, true_df], ignore_index=True)

In [14]:
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["label"].value_counts())

Dataset shape: (44898, 5)

Columns: ['title', 'text', 'subject', 'date', 'label']

Label distribution:
label
0    23481
1    21417
Name: count, dtype: int64


In [16]:
def message_preprocess(msg):
    msg = msg.lower()
    msg = re.sub(r'[^a-z\s]','', msg)
    msg = re.sub(r'\s+',' ', msg)
    msg = msg.strip()
    return msg

In [17]:
df["text"] = df["text"].apply(message_preprocess)

In [18]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [19]:
vectorizer = TfidfVectorizer(
    stop_words="english",
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [20]:
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [21]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:",
      accuracy_score(y_test, y_pred) * 100)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 94.83296213808462

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      5871
           1       0.93      0.96      0.95      5354

    accuracy                           0.95     11225
   macro avg       0.95      0.95      0.95     11225
weighted avg       0.95      0.95      0.95     11225


Confusion Matrix:
[[5502  369]
 [ 211 5143]]


In [22]:
def predict_news():
    news = input("Enter a news article: ")

    clean_news = message_preprocess(news)

    news_tfidf = vectorizer.transform([clean_news])

    prediction = model.predict(news_tfidf)[0]

    result = "Genuine" if prediction == 1 else "Fake"

    print("Predicted News:", result)

In [24]:
predict_news()

Predicted News: Genuine
